# 05 · Preparación Gold: BO-VMD y features

| | |
|---|---|
| **Fase CRISP-DM** | 3 — Preparación de los datos |
| **Objetivo** | Elegir los hiperparámetros de VMD por optimización bayesiana, filtrar las series (train: por segmento; validación: causal por origen), calcular el ancla, el objetivo normalizado y las covariables. |
| **Entradas** | `data/silver/voltajes_silver.parquet`, `data/silver/split.json` |
| **Salidas** | `data/gold/series_filtradas.parquet`, `data/gold/origenes_validacion.parquet`, `data/gold/escaladores.json`, `data/gold/_metadata.json`, `models/vmd_params.json`, figuras en `reports/gold/` |

**Todo ajuste usa solo train** (hasta la fecha de corte de `split.json`). La serie completa nunca se descompone: en validación, VMD se aplica por cada origen `t0` solo sobre los últimos `VMD_HISTORY` días hasta `t0`.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))
# Los procesos de joblib también necesitan encontrar el paquete
os.environ["PYTHONPATH"] = os.pathsep.join([str(ROOT / "src"), os.environ.get("PYTHONPATH", "")])

In [ ]:
import json
import time
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed

from voltcast import cleaning as cl
from voltcast import config, io
from voltcast import dataset as ds
from voltcast import vmd

config.ensure_dirs()
FIG_DIR = config.REPORTS_DIR / "gold"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
COL_EL = {"A": "tab:blue", "B": "tab:orange"}


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")


print(f"BO: {config.BO_SAMPLE_CELLS} celdas × {config.BO_TRIALS} pruebas | K ∈ {vmd.K_RANGE}, alpha ∈ {vmd.ALPHA_RANGE} | "
      f"modos descartados: {vmd.N_DROP}")
print(f"L={config.LOOKBACK_DAYS} | H={config.HORIZON_DAYS} | ancla={config.ANCHOR_DAYS} días | "
      f"VMD_HISTORY={config.VMD_HISTORY} | EVAL_STRIDE={config.EVAL_STRIDE} | VMD_MIN_LEN={config.VMD_MIN_LEN}")

## 1. Lectura de Silver

In [ ]:
silver = io.read_layer("silver", "voltajes_silver.parquet")
split = io.read_json("silver", "split.json")
corte = pd.Timestamp(split["fecha_corte"])

CELLS = sorted(silver["celda"].astype(str).unique())
EL_OF = {c: c[0] for c in CELLS}
V = ds.pivot(silver, "voltaje", CELLS)                     # voltaje real (Silver)
Q = ds.pivot(silver, "calidad", CELLS)
OOS = Q == "fuera_servicio"
VALIDO = Q.isin(cl.CALIDAD_VALIDA)
fechas = V.index
es_train = fechas < corte
print(f"{len(CELLS)} celdas × {len(fechas)} días | train: {es_train.sum()} días (hasta {split['train_hasta']}) | "
      f"validación: {(~es_train).sum()} días (desde {split['val_desde']})")

## 2. Optimización bayesiana de VMD sobre una muestra representativa

Se eligen `BO_SAMPLE_CELLS` celdas (10 por electrolizador), estratificadas por **posición** en el stack (inicio, medio y final) y por **nivel** de voltaje en train (terciles). Se excluyen las celdas con tramos `fuera_servicio`.

Para cada celda, `optuna` (TPE) busca `K` ∈ [3, 10] y `alpha` ∈ [500, 5000] (escala logarítmica) que **minimizan la mínima entropía de envolvente** de los modos. La entropía de envolvente de un modo es E = −Σ p·ln p, con p la envolvente de Hilbert normalizada.

In [ ]:
elegibles = pd.DataFrame({
    "celda": CELLS,
    "electrolizador": [EL_OF[c] for c in CELLS],
    "posicion": [int(c[3:]) for c in CELLS],
    "nivel": V.loc[es_train].where(VALIDO.loc[es_train]).mean().reindex(CELLS).to_numpy(),
    "con_oos": OOS.any().reindex(CELLS).to_numpy(),
})
elegibles = elegibles[~elegibles["con_oos"]]
muestra = vmd.select_bo_sample(elegibles, config.BO_SAMPLE_CELLS, config.SEED)
print(f"{len(elegibles)} celdas elegibles (sin fuera_servicio). Muestra ({len(muestra)}):")
elegibles.set_index("celda").loc[muestra, ["electrolizador", "posicion", "nivel"]].round(3).T

In [ ]:
t = time.time()
serie_train = {c: V.loc[es_train, c].to_numpy() for c in muestra}
bo = Parallel(n_jobs=config.N_JOBS, max_nbytes=None)(
    delayed(vmd.bo_vmd)(serie_train[c], config.BO_TRIALS, config.SEED) for c in muestra)
print(f"BO-VMD: {len(muestra)} celdas × {config.BO_TRIALS} pruebas en {time.time() - t:.0f} s")

bo_res = pd.DataFrame([{"celda": c, "electrolizador": EL_OF[c], "posicion": int(c[3:]),
                        "K": r["K"], "alpha": r["alpha"], "entropia": r["entropia"]} for c, r in zip(muestra, bo)])
bo_trials = pd.concat([r["trials"].assign(celda=c) for c, r in zip(muestra, bo)], ignore_index=True)
bo_res.round({"alpha": 0, "entropia": 3})

### Verificación de homogeneidad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for el in ["A", "B"]:
    r = bo_res[bo_res["electrolizador"] == el]
    axes[0].scatter(r["K"] + np.random.default_rng(0).uniform(-0.15, 0.15, len(r)), r["alpha"], s=50,
                    color=COL_EL[el], label=f"EL {el}", alpha=0.8)
axes[0].set(yscale="log", xlabel="K óptimo", ylabel="alpha óptimo", title="Óptimos por celda de la muestra")
axes[0].axhline(vmd.ALPHA_RANGE[0], ls=":", color="grey"); axes[0].legend()
sns.boxplot(data=bo_res, x="electrolizador", y="K", ax=axes[1], hue="electrolizador", palette=COL_EL, legend=False)
axes[1].set(title="K óptimo por electrolizador")
# Todas las pruebas: entropía frente a alpha, coloreada por K
sc = axes[2].scatter(bo_trials["params_alpha"], bo_trials["value"], c=bo_trials["params_K"], cmap="viridis", s=8)
axes[2].set(xscale="log", xlabel="alpha", ylabel="mín. entropía de envolvente", title="Todas las pruebas de la BO")
fig.colorbar(sc, ax=axes[2], label="K")
fig.tight_layout(); savefig(fig, "01_bo_homogeneidad")

bo_res.groupby("electrolizador")[["K", "alpha", "entropia"]].agg(["median", "min", "max"]).round(2)

In [ ]:
decision = vmd.choose_params(bo_res)
en_borde = float((bo_res["alpha"] < vmd.ALPHA_RANGE[0] * 1.1).mean())
print(f"Modo: {decision['modo']} | Δ mediana K = {decision['diferencia_K']:.1f} | "
      f"cociente de alpha = {decision['cociente_alpha']:.2f}")
for el, p in decision["params"].items():
    print(f"   EL {el}: K = {p['K']}, alpha = {p['alpha']:.0f}")
print(f"Celdas con alpha óptimo a < 10 % del límite inferior ({vmd.ALPHA_RANGE[0]:.0f}): {en_borde:.0%}")

vmd_params = {
    **decision,
    "objetivo": "min(entropía de envolvente de los modos)",
    "K_range": vmd.K_RANGE, "alpha_range": vmd.ALPHA_RANGE,
    "tau": vmd.TAU, "DC": vmd.DC, "init": vmd.INIT, "tol": vmd.TOL,
    "alpha_en_borde_inferior": en_borde,
    "muestra": bo_res.to_dict(orient="records"),
    "entrenado_hasta": split["train_hasta"],
}
PARAMS = {el: (p["K"], p["alpha"]) for el, p in decision["params"].items()}
(config.MODELS_DIR / "vmd_params.json").write_text(json.dumps(vmd_params, indent=2, ensure_ascii=False, default=float),
                                                    encoding="utf-8")
print(f"✔ {(config.MODELS_DIR / 'vmd_params.json').relative_to(config.BASE_DIR)}")

**Decisión.** Los óptimos de la muestra son parecidos entre electrolizadores: la mediana de `K` es 4 en el EL A y 3,5 en el EL B, y la de `alpha` ≈ 625 y ≈ 575. Cumplen el criterio de homogeneidad (|ΔK| ≤ 1 y cociente de alpha ≤ 2), así que se usa **un par común**: la mediana de la muestra, **K = 4 y alpha ≈ 620**. Se descarta **1 modo**, el de mayor frecuencia central.

**Advertencia.** En ~40 % de las celdas el `alpha` óptimo queda pegado al límite inferior del rango (500). La mínima entropía de envolvente favorece modos de banda ancha (`alpha` bajo), así que el criterio empuja hacia un filtrado **suave**. Con el rango de la especificación, la señal filtrada difiere de la real en ~1 mV de mediana (sección 3.1): VMD elimina el ruido de día a día, pero conserva la tendencia y los saltos de nivel. Bajar `alpha` aún más quitaría todavía menos ruido, por eso se mantiene el rango especificado.

Además, el gráfico de todas las pruebas muestra una superficie casi **plana y ruidosa**: la entropía varía en un rango parecido (≈ 6,1–6,8) para cualquier `alpha` y `K`. La BO distingue poco entre configuraciones, así que el par elegido es razonable, pero no un óptimo marcado. Para la PoC basta; si el modelo resultara sensible al filtrado, convendría probar un criterio orientado a la predicción (p. ej., el error de validación del modelo).

## 3. Filtrado

In [ ]:
# Ejemplo: descomposición de una celda de la muestra (último año de train)
c = muestra[0]
K, alpha = PARAMS[EL_OF[c]]
x = V.loc[es_train, c].to_numpy()
u = vmd.decompose(vmd.fill_gaps(x), K, alpha)
idx = fechas[es_train]
ult = slice(-365, None)
fig, axes = plt.subplots(K + 1, 1, figsize=(14, 1.6 * (K + 1)), sharex=True)
axes[0].plot(idx[ult], x[ult], color="lightgrey", label="real (Silver)")
axes[0].plot(idx[ult], u[: K - vmd.N_DROP].sum(0)[ult], color="k", lw=1, label=f"filtrada (sin {vmd.N_DROP} modo)")
axes[0].legend(loc="upper left", fontsize=8); axes[0].set_title(f"{c}: VMD con K={K}, alpha={alpha:.0f}")
for k in range(K):
    axes[k + 1].plot(idx[ult], u[k][ult], lw=0.8, color="tab:red" if k >= K - vmd.N_DROP else "tab:blue")
    axes[k + 1].set_ylabel(f"modo {k + 1}", fontsize=8)
fig.tight_layout(); savefig(fig, "02_modos_ejemplo")

### 3.1 Train: VMD sobre el segmento de train de cada celda

Cada celda se filtra sobre sus **segmentos en servicio** dentro de train (los tramos `fuera_servicio` cortan la serie). Los segmentos más cortos que `VMD_MIN_LEN` días quedan sin filtrar (`NaN`) y no generan muestras. Los huecos internos se interpolan solo para alimentar a VMD; el canal-máscara del modelo los identifica.

In [ ]:
t = time.time()
Vtr, OOStr = V.loc[es_train], OOS.loc[es_train]
filtrada = Parallel(n_jobs=config.N_JOBS, max_nbytes=None)(
    delayed(vmd.filter_segments)(Vtr[c].to_numpy(), OOStr[c].to_numpy(), *PARAMS[EL_OF[c]]) for c in CELLS)
VF = pd.DataFrame(np.column_stack(filtrada), index=Vtr.index, columns=CELLS)
print(f"VMD de train: {len(CELLS)} celdas en {time.time() - t:.0f} s")

resid = (Vtr - VF).where(VALIDO.loc[es_train])
print(f"Real − filtrada (fechas válidas): RMSE {np.sqrt(np.nanmean(resid.to_numpy() ** 2)) * 1000:.1f} mV | "
      f"P95 |Δ| {np.nanpercentile(np.abs(resid.to_numpy()), 95) * 1000:.1f} mV")
print(f"Celdas×día de train sin filtrar (fuera de servicio o segmento corto): {VF.isna().to_numpy().mean():.2%}")

### 3.2 Validación: VMD causal por origen

Orígenes `t0` cada `EVAL_STRIDE` días, desde el último día de train hasta el penúltimo de validación. Para cada origen y celda, VMD se aplica solo sobre los últimos `VMD_HISTORY` días **hasta `t0`** (el segmento en servicio que termina en `t0`), y se guardan los últimos `LOOKBACK_DAYS` días filtrados como ventana de entrada.

In [ ]:
origenes = ds.validation_origins(split, config.EVAL_STRIDE)
pos_t0 = fechas.get_indexer(origenes)
print(f"{len(origenes)} orígenes: {origenes[0].date()} → {origenes[-1].date()} "
      f"→ {len(origenes) * len(CELLS):,} descomposiciones")


def ventanas_celda(c):
    x, b = V[c].to_numpy(), OOS[c].to_numpy()
    K, alpha = PARAMS[EL_OF[c]]
    return np.stack([vmd.causal_window(x, b, t0, K, alpha) for t0 in pos_t0])  # [n_origenes, L]


t = time.time()
W = np.stack(Parallel(n_jobs=config.N_JOBS, max_nbytes=None)(delayed(ventanas_celda)(c) for c in CELLS), axis=1)  # [O, C, L]
print(f"VMD causal: {W.shape} en {time.time() - t:.0f} s | ventanas completas: {np.isfinite(W).all(-1).mean():.1%}")

### 3.3 Diagnóstico: filtrado causal frente al filtrado de train

En train la ventana de entrada sale de una VMD sobre todo el segmento (usa días posteriores a cada `t0`); en validación e inferencia, de una VMD causal que termina en `t0`, donde VMD sufre efecto de borde. Para medir esa diferencia se repite el filtrado causal sobre orígenes del **último año de train** en las celdas de la muestra y se compara el **ancla** resultante.

In [ ]:
pos_diag = np.arange(es_train.sum() - 365, es_train.sum(), 14)


def diag_celda(c):
    x, b = V[c].to_numpy(), OOS[c].to_numpy()
    K, alpha = PARAMS[EL_OF[c]]
    return [ds.anchor_from_window(vmd.causal_window(x, b, t0, K, alpha)) for t0 in pos_diag]


anc_causal = np.array(Parallel(n_jobs=config.N_JOBS, max_nbytes=None)(delayed(diag_celda)(c) for c in muestra)).T  # [t0, celda]
anc_train = ds.anchor(VF[muestra]).to_numpy()[pos_diag]
anc_real = ds.anchor(V[muestra].where(VALIDO[muestra])).to_numpy()[pos_diag]
dif = pd.DataFrame({
    "causal − train": (anc_causal - anc_train).ravel(),
    "train − real": (anc_train - anc_real).ravel(),
    "causal − real": (anc_causal - anc_real).ravel(),
}).dropna() * 1000
dif.abs().describe().loc[["count", "mean", "50%", "max"]].round(2).rename(index={"mean": "MAE (mV)", "50%": "mediana (mV)", "max": "máx (mV)"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for col in dif:
    ax.hist(dif[col], bins=40, alpha=0.5, label=col)
ax.set(xlabel="diferencia de ancla (mV)", ylabel="orígenes × celdas", title="Ancla: filtrado causal vs. filtrado de train")
ax.legend(); fig.tight_layout(); savefig(fig, "03_causal_vs_train")

**Lectura.** El ancla obtenida con VMD causal difiere de la de train en ~0,5 mV de media (máximo ≈ 8 mV). Es despreciable frente a σ_Δ ≈ 129 mV. Por lo tanto, el efecto de borde de VMD y la diferencia entre el filtrado de train (no causal) y el de validación (causal) **no introducen un sesgo relevante** en el ancla ni en el objetivo. Se mantiene el esquema de la especificación.

## 4. Ancla, objetivo normalizado y σ_Δ (solo train)

In [ ]:
ANC = ds.anchor(VF)
ok_t0 = ds.origin_ok(ANC, OOStr)
sigma_delta, por_h = ds.delta_stats(Vtr, VALIDO.loc[es_train], ANC, ok_t0, config.HORIZON_DAYS)
print(f"σ_Δ (global, train) = {sigma_delta * 1000:.1f} mV sobre {por_h['n'].sum():,} pares (celda, t0, h)")
print(f"Orígenes de train usables: {ok_t0.to_numpy().sum():,} celda×día")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(por_h.index, por_h["desvio"] * 1000, label="desvío de Δ(h)")
ax.plot(por_h.index, por_h["media"] * 1000, label="media de Δ(h)")
ax.axhline(sigma_delta * 1000, ls="--", color="grey", label="σ_Δ global")
for h in config.EVAL_HORIZONS:
    ax.axvline(h, ls=":", color="k")
ax.set(xlabel="horizonte h (días)", ylabel="mV", title="Δ(t0, h) = V_real(t0 + h) − ancla(t0) en train"); ax.legend()
fig.tight_layout(); savefig(fig, "04_delta_por_horizonte")
por_h.loc[[1, 7, 30, 60, 90, 180]].assign(media=lambda d: d["media"] * 1000, desvio=lambda d: d["desvio"] * 1000).round(1)

## 5. Covariables del electrolizador

`kA` y `V_total / n_en_servicio` (voltaje medio por celda en servicio, regla R7), estandarizadas con media y desvío de **train sin días de paro** y acotadas a ±`COV_CLIP`. Si se incluyeran los paros, el desvío se inflaría (kA pasaría de ≈ 0,3 a ≈ 1,8) y la variación normal quedaría comprimida. Los paros quedan como valores extremos (≈ −`COV_CLIP` en kA), así el modelo los reconoce sin que dominen la escala.

In [ ]:
cov = (silver.groupby(["fecha", "electrolizador"], observed=True)[["kA", "V_total", "n_en_servicio"]].first()
       .reset_index())
# [Volts] suma todas las celdas físicas: las que no están en el Excel se suman a las celdas en servicio
n_excel = silver.groupby("electrolizador", observed=True)["celda"].nunique()
extra = (config.N_CELDAS_FISICAS - cov["electrolizador"].map(n_excel).astype(int)) if config.N_CELDAS_FISICAS else 0
cov["V_celda_media"] = cov["V_total"] / (cov["n_en_servicio"] + extra)
# Estadísticos de train sin días de paro (los paros inflarían el desvío); z acotado a ±COV_CLIP
tr = (cov["fecha"] < corte) & (cov["kA"] >= config.KA_SHUTDOWN)
stats = {v: {"media": float(cov.loc[tr, v].mean()), "desvio": float(cov.loc[tr, v].std())} for v in ["kA", "V_celda_media"]}
for v, z in [("kA", "kA_z"), ("V_celda_media", "vcel_z")]:
    cov[z] = ((cov[v] - stats[v]["media"]) / stats[v]["desvio"]).clip(-config.COV_CLIP, config.COV_CLIP)
print(f"z acotado a ±{config.COV_CLIP}: kA en {(cov['kA_z'].abs() == config.COV_CLIP).mean():.1%} de los días, "
      f"V_celda_media en {(cov['vcel_z'].abs() == config.COV_CLIP).mean():.1%}")

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
for el in ["A", "B"]:
    g = cov[cov["electrolizador"] == el]
    axes[0].plot(g["fecha"], g["kA_z"], lw=0.7, color=COL_EL[el], label=f"EL {el}")
    axes[1].plot(g["fecha"], g["vcel_z"], lw=0.7, color=COL_EL[el], label=f"EL {el}")
for ax, t_ in zip(axes, ["kA (z)", "V_total / n_en_servicio (z)"]):
    ax.axvline(corte, color="k", ls="--"); ax.set(ylabel=t_); ax.legend(loc="lower left")
fig.tight_layout(); savefig(fig, "05_covariables")
pd.DataFrame(stats).round(4)

## 6. Escritura en Gold

In [ ]:
# Serie diaria por celda: real, calidad, filtrada (solo train) y covariables estandarizadas
gold = silver[["fecha", "electrolizador", "celda", "voltaje", "calidad"]].copy()
gold["celda"] = gold["celda"].astype(str)
vf_largo = VF.rename_axis("fecha").reset_index().melt(id_vars="fecha", var_name="celda", value_name="v_filtrada")
gold = gold.merge(vf_largo, on=["fecha", "celda"], how="left", validate="one_to_one")
gold = gold.merge(cov[["fecha", "electrolizador", "kA_z", "vcel_z"]], on=["fecha", "electrolizador"], how="left",
                  validate="many_to_one")
gold["fuera_servicio"] = gold["calidad"] == "fuera_servicio"
gold["split"] = np.where(gold["fecha"] < corte, "train", "val")
for col in ["electrolizador", "celda", "split"]:
    gold[col] = gold[col].astype("category")

# Ventanas causales de validación en formato largo
O, C, L = W.shape
offs = np.arange(-L + 1, 1)
orig_largo = pd.DataFrame({
    "fecha_origen": np.repeat(origenes, C * L),
    "celda": np.tile(np.repeat(CELLS, L), O),
    "offset": np.tile(offs, O * C).astype(np.int16),
    "v_filtrada": W.ravel(),
})
orig_largo["fecha"] = orig_largo["fecha_origen"] + pd.to_timedelta(orig_largo["offset"], unit="D")
orig_largo["electrolizador"] = orig_largo["celda"].str[0].astype("category")
orig_largo["celda"] = orig_largo["celda"].astype("category")

escaladores = {
    "sigma_delta": sigma_delta,
    "covariables": stats, "cov_clip": config.COV_CLIP,
    "anchor_days": config.ANCHOR_DAYS, "lookback_days": config.LOOKBACK_DAYS, "horizon_days": config.HORIZON_DAYS,
    "ajustado_hasta": split["train_hasta"],
    "vmd_params": {el: {"K": k, "alpha": a} for el, (k, a) in PARAMS.items()}, "vmd_n_drop": vmd.N_DROP,
}
src = config.SILVER_DIR / "voltajes_silver.parquet"
src_hash = io.file_sha256(src)
extra = {"split": split}
for df, name in [(gold, "series_filtradas.parquet"), (orig_largo, "origenes_validacion.parquet")]:
    p = io.write_layer(df, "gold", name, source=src, source_hash=src_hash, extra=extra)
    print(f"✔ {p.relative_to(config.BASE_DIR)} ({len(df):,} filas, {p.stat().st_size / 1e6:.1f} MB)")
p = io.write_json(escaladores, "gold", "escaladores.json")
print(f"✔ {p.relative_to(config.BASE_DIR)}")

## 7. Verificaciones

In [ ]:
g = io.read_layer("gold", "series_filtradas.parquet")
o = io.read_layer("gold", "origenes_validacion.parquet")
e = io.read_json("gold", "escaladores.json")
checks = {
    "sin filtrada en fechas de validación (no se descompone la serie completa)": g.loc[g["split"] == "val", "v_filtrada"].isna().all(),
    "ventanas causales: todas las fechas ≤ t0": (o["fecha"] <= o["fecha_origen"]).all(),
    "primer origen = último día de train": o["fecha_origen"].min() == pd.Timestamp(split["train_hasta"]),
    "una fila por (celda, fecha)": not g.duplicated(["celda", "fecha"]).any(),
    "una fila por (origen, celda, offset)": not o.duplicated(["fecha_origen", "celda", "offset"]).any(),
    "σ_Δ positivo": e["sigma_delta"] > 0,
    "covariables sin NaN en fechas válidas": g.loc[g["calidad"].isin(cl.CALIDAD_VALIDA), ["kA_z", "vcel_z"]].notna().all().all(),
}
for k, v in checks.items():
    print(("✔" if v else "✘"), k)
assert all(checks.values()), "Alguna verificación de Gold falló"

## Conclusión

- **VMD:** un par común para los dos electrolizadores (`models/vmd_params.json`), elegido por BO sobre 20 celdas. El filtrado resultante es suave (~1 mV de mediana respecto de la señal real).
- **Filtrado sin fuga de información:** en train se filtra por segmento en servicio; en validación, de forma causal por origen (70 orígenes × 300 celdas). La serie completa nunca se descompone.
- **Objetivo:** `y_h = (V_real(t0 + h) − ancla) / σ_Δ`, con σ_Δ ≈ 129 mV. El desvío de Δ crece con el horizonte, de ≈ 20 mV a 1 día a ≈ 108 mV a 60 días y ≈ 178 mV a 180 días. Esa es la escala contra la que se medirá el error del modelo y de los baselines. La media de Δ se vuelve negativa con el horizonte (≈ −18 mV a 180 días): las caídas por intervenciones pesan más que la subida lenta por degradación.
- **Covariables:** `kA_z` y `vcel_z` por electrolizador y día, estandarizadas con train sin paros.

**Siguiente:** notebook 06 — Modelado (Bi-NOA-LSTM y ablación con Bi-LSTM estándar).